SEMANTIC EVENT EMBEDDINGS + LSTM

message
 embedding
 sequence
 LSTM
 next embedding prediction

In [1]:
# Parsing logs

import csv
import json

dataset = []

with open("graylog_1.csv", newline="", encoding="utf-8") as f:

    reader = csv.DictReader(f)

    for row in reader:

        try:

            log_data = json.loads(row["log"])

            event = {

                "timestamp":
                    row["@timestamp"],

                "request_id":
                    log_data.get("request_id"),

                "service":
                    log_data.get("service_name"),

                "level":
                    log_data.get("level"),

                "message":
                    log_data.get("full_message")
            }

            dataset.append(event)

        except Exception:

            continue

print(len(dataset))

788092


In [2]:
# External logs added
import pandas as pd
import json

public_dataset = []

df = pd.read_csv("OpenStack_2k.log_structured.csv")

print(df.columns)
print(df.head())

Index(['LineId', 'Logrecord', 'Date', 'Time', 'Pid', 'Level', 'Component',
       'ADDR', 'Content', 'EventId', 'EventTemplate'],
      dtype='str')
   LineId                           Logrecord        Date          Time  \
0       1  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:00.008   
1       2  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:00.272   
2       3  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:01.551   
3       4  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:01.813   
4       5  nova-api.log.1.2017-05-16_13:53:08  2017-05-16  00:00:03.091   

     Pid Level                       Component  \
0  25746  INFO  nova.osapi_compute.wsgi.server   
1  25746  INFO  nova.osapi_compute.wsgi.server   
2  25746  INFO  nova.osapi_compute.wsgi.server   
3  25746  INFO  nova.osapi_compute.wsgi.server   
4  25746  INFO  nova.osapi_compute.wsgi.server   

                                                ADDR  \
0  req-38101a0b-2096-447d-96ea-a692162415ae

In [3]:
for _, row in df.iterrows():

    event = {

        "timestamp":
            row.get("Time", ""),

        "request_id":
            str(row.get("Pid", "unknown")),

        "service":
            "openstack-service",

        "level":
            row.get("Level", "INFO"),

        "message":
            row.get("Content", "")
    }

    public_dataset.append(event)

print(len(public_dataset))
print(public_dataset[0])

2000
{'timestamp': '00:00:00.008', 'request_id': '25746', 'service': 'openstack-service', 'level': 'INFO', 'message': '10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2477829'}


In [4]:
dataset.extend(public_dataset)

In [5]:
# Build traces

from collections import defaultdict

traces = defaultdict(list)

for event in dataset:

    rid = event["request_id"]

    if rid:

        traces[rid].append(event)

for trace in traces.values():

    trace.sort(
        key=lambda x: x["timestamp"]
    )

print(len(traces))

100022


In [6]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
from sentence_transformers import SentenceTransformer

In [8]:
embedder = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
messages = [
    f"{e['service']} {e['level']} {e['message']}"
    # e["message"]

    for e in dataset
]

embeddings = embedder.encode(

    messages,

    show_progress_bar=True
)
#
# for event in dataset:
#
#     embedding = embedder.encode(
#         event["message"]
#     )
#
#     event["embedding"] = embedding

Batches:   0%|          | 0/24691 [00:00<?, ?it/s]

In [10]:
for event, emb in zip(

    dataset,
    embeddings
):

    event["embedding"] = emb

print(
    dataset[0]["embedding"].shape
)

(384,)


In [11]:
from collections import defaultdict

traces = defaultdict(list)

for event in dataset:

    rid = event["request_id"]

    if rid:

        traces[rid].append(event)

for trace in traces.values():

    trace.sort(
        key=lambda x: x["timestamp"]
    )

In [12]:
WINDOW_SIZE = 3

X_embed = []
y_embed = []

for trace in traces.values():

    seq = [

        e["embedding"]

        for e in trace
    ]

    if len(seq) <= WINDOW_SIZE:
        continue

    for i in range(

        len(seq) - WINDOW_SIZE

    ):

        X_embed.append(

            seq[i:i+WINDOW_SIZE]
        )

        y_embed.append(

            seq[i+WINDOW_SIZE]
        )


In [13]:
print(len(X_embed))

490028


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(

    X_embed,
    y_embed,

    test_size=0.2,

    random_state=42
)

In [15]:
import torch
from torch.utils.data import Dataset

class EmbeddingDataset(Dataset):

    def __init__(

        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [16]:
from torch.utils.data import DataLoader

train_loader = DataLoader(

    EmbeddingDataset(
        X_train,
        y_train
    ),

    batch_size=32,

    shuffle=True
)

val_loader = DataLoader(

    EmbeddingDataset(
        X_val,
        y_val
    ),

    batch_size=32
)

/var/folders/0n/rz6tvdmd76391q884vkcgbhr0000gn/T/ipykernel_14888/1147821585.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  self.X = torch.tensor(


In [17]:
import torch.nn as nn

class SemanticLSTM(nn.Module):

    def __init__(

        self,
        embedding_dim=384,
        hidden_size=128
    ):

        super().__init__()

        self.lstm = nn.LSTM(

            embedding_dim,

            hidden_size,

            batch_first=True
        )

        self.fc = nn.Linear(

            hidden_size,

            embedding_dim
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [18]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

model = SemanticLSTM().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=1e-3
)

In [19]:
EPOCH = 10

In [20]:
for epoch in range(EPOCH):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(

            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(

        f"Epoch {epoch+1} "
        f"Loss={total_loss:.4f}"
    )

Epoch 1 Loss=0.7930
Epoch 2 Loss=0.6953
Epoch 3 Loss=0.6887
Epoch 4 Loss=0.6858
Epoch 5 Loss=0.6841
Epoch 6 Loss=0.6828
Epoch 7 Loss=0.6821
Epoch 8 Loss=0.6814
Epoch 9 Loss=0.6808
Epoch 10 Loss=0.6809


In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

all_embeddings = np.array(

    [e["embedding"] for e in dataset]
)

all_messages = [

    e["message"]
    for e in dataset
]

def find_closest_message(vec):

    sims = cosine_similarity(

        [vec],
        all_embeddings
    )[0]

    idx = sims.argmax()

    return (

        all_messages[idx],
        sims[idx]
    )

In [22]:
model.eval()

sample = torch.tensor(

    [X_val[0]],

    dtype=torch.float32
).to(device)

with torch.no_grad():

    pred_vec = model(sample)

pred_vec = pred_vec.cpu().numpy()[0]

real_msg, _ = find_closest_message(

    y_val[0]
)

pred_msg, score = find_closest_message(

    pred_vec
)

print("\nREAL NEXT EVENT:\n")

print(real_msg)

print("\nPREDICTED NEXT EVENT:\n")

print(pred_msg)

print("\nSIMILARITY:\n")

print(score)


REAL NEXT EVENT:

Processing payment

PREDICTED NEXT EVENT:

Processing payment

SIMILARITY:

0.9999838


In [23]:
msg1 = "database timeout"

msg2 = "connection refused"

msg3 = "user login success"

e1 = embedder.encode(msg1)

e2 = embedder.encode(msg2)

e3 = embedder.encode(msg3)

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
print(
    cosine_similarity(
        [e1],
        [e2]
    )
)

print(
    cosine_similarity(
        [e1],
        [e3]
    )
)

[[0.2942071]]
[[0.19770652]]


In [26]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

ANOMALY_THRESHOLD = 0.7

In [27]:
def detect_anomaly(trace):

    seq = [

        e["embedding"]

        for e in trace
    ]

    if len(seq) < WINDOW_SIZE + 1:

        return None

    anomalies = []

    for i in range(

        len(seq) - WINDOW_SIZE

    ):

        window = seq[i:i+WINDOW_SIZE]

        real_next = seq[i+WINDOW_SIZE]

        x = torch.tensor(

            [window],

            dtype=torch.float32
        ).to(device)

        with torch.no_grad():

            pred = model(x)

        pred = pred.cpu().numpy()[0]

        similarity = cosine_similarity(

            [pred],
            [real_next]
        )[0][0]

        if similarity < ANOMALY_THRESHOLD:

            anomalies.append({

                "position": i,

                "similarity": similarity,

                "expected":
                    find_closest_message(pred)[0],

                "actual":
                    find_closest_message(real_next)[0]
            })

    return anomalies

In [30]:
sample_trace = list(traces.values())[2]

anomalies = detect_anomaly(
    sample_trace
)

print(anomalies)

[]


In [31]:
torch.save(
    model.state_dict(),
    "semantic_lstm.pt"
)